# Unintended Memorization in LLMs — Reproducible Colab Launcher

Pipeline modulare e riproducibile per lo studio empirico su **`Qwen/Qwen2.5-0.5B-Instruct`**.

**Istruzioni rapide su Google Colab (Runtime → Change runtime type → T4 GPU):**
1. Carica questo file (`run_colab.zip`).
2. Imposta `USE_DRIVE` nella Cella 2 (opzionale).
3. Esegui **Runtime → Run all**: tutte le celle sono progettate per completarsi senza interventi manuali.

In [ ]:
# 1. Clona il repository, imposta la working directory, rimuovi conflitto torchao e installa dipendenze
import sys
from pathlib import Path

if not Path("unintended-memorization").exists():
    !git clone https://github.com/giuseppedima/unintended-memorization.git

%cd /content/unintended-memorization/code
print('Working directory:', Path.cwd())
sys.path.insert(0, str((Path.cwd() / "src").resolve()))

!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt
!pip install -q --no-deps -e .

import torch, transformers, peft, datasets, accelerate
print(
    f'torch={torch.__version__} | transformers={transformers.__version__} | '
    f'peft={peft.__version__} | datasets={datasets.__version__} | '
    f'accelerate={accelerate.__version__}'
)

In [ ]:
# 2. (Opzionale) Monta Google Drive e sincronizza automaticamente USE_GOOGLE_DRIVE in .env
import re
from pathlib import Path

USE_DRIVE = True  # Imposta True per salvare automaticamente su Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

env_path = Path('.env')
env_txt = env_path.read_text(encoding='utf-8')
env_txt = re.sub(
    r'^USE_GOOGLE_DRIVE=.*$',
    f"USE_GOOGLE_DRIVE={'true' if USE_DRIVE else 'false'}",
    env_txt,
    flags=re.MULTILINE,
)
env_path.write_text(env_txt, encoding='utf-8')
print('USE_GOOGLE_DRIVE synced to:', USE_DRIVE)


In [ ]:
# 3. Verifica configurazione corrente (.env)
from pathlib import Path
print(Path('.env').read_text(encoding='utf-8'))


In [ ]:
# 4. Esecuzione della suite di test unitari (pytest)
!pytest -v


In [ ]:
# 5. Stage 1 — Costruzione corpus 256x96 token IDs + Sanity Checks SC-1..SC-5
!python scripts/run_pipeline.py --stage corpus


In [ ]:
# 6. Stage 2 & 3 — Valutazione Pre-Training, LoRA Fine-Tuning (2 epoche) e Valutazione Post-Training
!python scripts/run_pipeline.py --stage train


In [ ]:
# 7. Stage 4 — Tabelle descrittive, verifica H1-H5 e generazione grafici
!python scripts/run_pipeline.py --stage analyze


In [ ]:
# 8. Visualizzazione inline dei grafici generati
from IPython.display import Image, display
from memorization.config import load_config

cfg = load_config()
for img_path in sorted(cfg.plots_dir.glob('*.png')):
    print(f'--- {img_path.name} ---')
    display(Image(filename=str(img_path)))


In [ ]:
# 9. Ispezione ordinata delle 48 continuazioni grezze post-training
import pandas as pd
from memorization.config import load_config

cfg = load_config()
df = pd.read_csv(cfg.results_dir / 'eval_paired_48.csv')
cols = [
    'group', 'cumulative_exposures', 'prompt_type', 'project_id',
    'secret', 'first_line_norm', 'strict_exact_match',
    'error_categories', 'net_delta_ll', 'w1_rank',
]
df.sort_values(['prompt_type', 'cumulative_exposures', 'canary_id'])[cols]
